# Experimento C: Validación de Integridad Metodológica (The Anti-Leakage Test)

## Objetivo
Demostrar empíricamente cómo el **data leakage** infla las métricas y **cuantificar el impacto de cada fuente** por separado.

## Modelos
- **Logistic Regression**, **Random Forest**, **XGBoost** (mismo modelo en todas las ramas)

## Cinco Ramas Experimentales
| Rama | Split | Escalado | SMOTE | Fuentes de leakage |
|------|-------|----------|-------|--------------------|
| **Correcta** | Temporal | Solo train | Solo train | 0 |
| **Leak_split** | Aleatorio | Solo train | Solo train | 1 (split) |
| **Leak_scaler** | Temporal | Global (train+test) | Solo train | 1 (escalado) |
| **Leak_smote** | Temporal | Solo train | Global (train+test) | 1 (SMOTE) |
| **Leak_todas** | Aleatorio | Global | Global | 3 |

## Parámetros SMOTE (config.py)
- `k_neighbors=5`, `sampling_strategy='auto'`, `random_state=SEED`

## Métricas
- **AUPRC**, **AUC ROC**, **Card Precision@100** (donde aplica)

In [ ]:
import os
import sys
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, RESULTS_DIR, FIGURES_DIR, COLORS, SMOTE_PARAMS,
)
from experiments.data_utils import load_transformed_data
from experiments.experiment_c_leakage_test import run_all

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

print("=" * 70)
print("  EXPERIMENTO C: ANTI-LEAKAGE TEST (REFACTORIZADO)")
print("=" * 70)
print(f"  Modelos: Logistic Regression, Random Forest, XGBoost")
print(f"  Ramas: Correcta, Leak_split, Leak_scaler, Leak_smote, Leak_todas")
print(f"  SMOTE: k_neighbors={SMOTE_PARAMS['k_neighbors']}, "
      f"sampling_strategy={SMOTE_PARAMS['sampling_strategy']}")
print(f"  Métricas: AUPRC, AUC ROC, Card Precision@100")

---
## 1. Carga de Datos

In [ ]:
transactions_df = load_transformed_data()
print(f"Dataset: {len(transactions_df):,} transacciones")

results, train_df, test_df = run_all(transactions_df)
print("\n✓ Ejecución completada: 5 ramas × 3 modelos")

---
## 2. Resultados por rama y modelo

In [ ]:
ramas_order = ["Correcta", "Leak_split", "Leak_scaler", "Leak_smote", "Leak_todas"]
models = ["Logistic Regression", "Random Forest", "XGBoost"]

rows = []
for model in models:
    for rama in ramas_order:
        r = results[model][rama]
        rows.append({'Modelo': model, 'Rama': rama, 'AUC ROC': r['auc_roc'], 'AUPRC': r['auprc'],
                     'CP@100': r.get('cp100', np.nan)})
df_all = pd.DataFrame(rows)
display(df_all.pivot(index='Modelo', columns='Rama', values=['AUC ROC', 'AUPRC']).round(4))

---
## 3. Desglose: impacto incremental por fuente de leakage (AUPRC)

In [ ]:
desglose_rows = []
for model in models:
    correct = results[model]["Correcta"]['auprc']
    desglose_rows.append({
        'Modelo': model,
        'Correcta': f"{correct:.4f}",
        'Leak_split (Δ)': f"{results[model]['Leak_split']['auprc']:.4f} ({results[model]['Leak_split']['auprc']-correct:+.4f})",
        'Leak_scaler (Δ)': f"{results[model]['Leak_scaler']['auprc']:.4f} ({results[model]['Leak_scaler']['auprc']-correct:+.4f})",
        'Leak_smote (Δ)': f"{results[model]['Leak_smote']['auprc']:.4f} ({results[model]['Leak_smote']['auprc']-correct:+.4f})",
        'Leak_todas (Δ)': f"{results[model]['Leak_todas']['auprc']:.4f} ({results[model]['Leak_todas']['auprc']-correct:+.4f})",
    })
df_desglose = pd.DataFrame(desglose_rows)
display(df_desglose)

---
## 4. Visualizaciones Comparativas

In [ ]:
# Gráfico: AUPRC por rama y modelo
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
x = np.arange(len(ramas_order))
colors_ramas = {'Correcta': COLORS['correct_pipeline'], 'Leak_split': '#FFA500',
                'Leak_scaler': '#FF8C00', 'Leak_smote': '#FF6347', 'Leak_todas': COLORS['incorrect_pipeline']}

for i, model in enumerate(models):
    ax = axes[i]
    vals = [results[model][r]['auprc'] for r in ramas_order]
    bars = ax.bar(x, vals, 0.6, color=[colors_ramas[r] for r in ramas_order], edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(['Correcta', 'Leak\nsplit', 'Leak\nscaler', 'Leak\nSMOTE', 'Leak\ntodas'], fontsize=9)
    ax.set_ylabel('AUPRC')
    ax.set_title(model)
    ax.set_ylim([0, 1.05])
    for bar, v in zip(bars, vals):
        ax.annotate(f'{v:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9)

fig.suptitle(f'Experimento C: Impacto del Data Leakage por fuente\n'
             f'SMOTE: k_neighbors={SMOTE_PARAMS["k_neighbors"]}, sampling_strategy={SMOTE_PARAMS["sampling_strategy"]}',
             fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_c_leakage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Figura: {FIGURES_DIR / 'experiment_c_leakage_comparison.png'}")

---
## 5. Tabla Comparativa: Correcto vs Incorrecto vs Experimento A

Requisito de la **Semana 1** del planning:  
> *"Comparar preliminarmente los resultados de A (Realista) vs C-Incorrecto (Inflado)"*  
> *"Elaborar tabla comparativa A vs C-Incorrecto"*

In [ ]:
# Tabla: C-Correcta vs C-Leak_todas por modelo
comparison_c = pd.DataFrame([
    {'Modelo': m, 'C-Correcta AUC': results[m]['Correcta']['auc_roc'],
     'C-Correcta AUPRC': results[m]['Correcta']['auprc'],
     'C-Correcta CP@100': results[m]['Correcta']['cp100'],
     'C-Leak_todas AUC': results[m]['Leak_todas']['auc_roc'],
     'C-Leak_todas AUPRC': results[m]['Leak_todas']['auprc'],}
    for m in models
])
display(comparison_c.round(4))

# Comparativa con Experimento A
exp_a_path = RESULTS_DIR / 'experiment_a_predictions.pkl'
if exp_a_path.exists():
    with open(exp_a_path, 'rb') as f:
        results_a = pickle.load(f)
    rows = []
    for model_name in models:
        res_a = results_a.get(model_name, {})
        auc_a = res_a.get('auc_roc', res_a.get('auc_roc_mean', np.nan))
        auprc_a = res_a.get('avg_precision', res_a.get('auprc_mean', np.nan))
        cp_a = res_a.get('card_precision_at_k', {}).get(100, np.nan)
        rows.append({'Experimento': f'A-Baseline: {model_name}', 'AUC ROC': auc_a, 'AUPRC': auprc_a, 'CP@100': cp_a})
    for model_name in models:
        r = results[model_name]['Correcta']
        rows.append({'Experimento': f'C-Correcta: {model_name}', 'AUC ROC': r['auc_roc'], 'AUPRC': r['auprc'], 'CP@100': r['cp100']})
    for model_name in models:
        r = results[model_name]['Leak_todas']
        rows.append({'Experimento': f'C-Leak_todas: {model_name} ⚠', 'AUC ROC': r['auc_roc'], 'AUPRC': r['auprc'], 'CP@100': np.nan})
    comparison_a_vs_c = pd.DataFrame(rows).set_index('Experimento')
    display(comparison_a_vs_c.round(4))
    comparison_a_vs_c.to_csv(RESULTS_DIR / 'experiment_c_vs_a_comparison.csv')
else:
    print("⚠ Ejecuta experiment_a_baseline para la comparativa A vs C.")

df_all.to_csv(RESULTS_DIR / 'experiment_c_all_ramas.csv', index=False)
df_desglose.to_csv(RESULTS_DIR / 'experiment_c_desglose_leakage.csv', index=False)
comparison_c.to_csv(RESULTS_DIR / 'experiment_c_comparison.csv', index=False)

---
## 6. Persistencia de Resultados

In [ ]:
# Guardar resultados completos
with open(RESULTS_DIR / 'experiment_c_results.pkl', 'wb') as f:
    pickle.dump({
        'results': results,
        'correct': {m: results[m]['Correcta'] for m in models},
        'incorrect': {m: results[m]['Leak_todas'] for m in models},
        'metadata': {'smote_params': SMOTE_PARAMS, 'ramas': ramas_order, 'models': models},
    }, f)

print("✓ Resultados guardados:")
print(f"  - PKL: {RESULTS_DIR / 'experiment_c_results.pkl'}")
print(f"  - CSV: {RESULTS_DIR / 'experiment_c_comparison.csv'}")
print(f"  - CSV: {RESULTS_DIR / 'experiment_c_desglose_leakage.csv'}")
print(f"  - FIG: {FIGURES_DIR / 'experiment_c_leakage_comparison.png'}")

---
## 7. Conclusiones del Experimento C

### Evidencia del Data Leakage

**Este experimento es la evidencia visual de la crítica al "Estado del Arte" deficiente.**

La diferencia entre ambas ramas demuestra cómo el Data Leakage infla artificialmente los resultados, dando una **falsa sensación de seguridad** sobre el rendimiento del modelo.

### Fuentes de Leakage Identificadas

1. **Escalado global**: `StandardScaler` ajustado con datos de test → la media y desviación incluyen información futura
2. **SMOTE global**: Las muestras sintéticas se generan interpolando vecinos que pueden pertenecer al periodo de test
3. **Split aleatorio**: Rompe la causalidad temporal, permitiendo que transacciones futuras entrenen al modelo

### Implicación para el TFM

Los resultados "inflados" de la rama incorrecta son comparables a los que publica parte de la literatura sobre detección de fraude.  
La rama correcta produce resultados **realistas**, coherentes con los del Experimento A (Baseline puro), y sirve como **validación de la integridad metodológica** del TFM.

### Relación con otros experimentos

- **vs Experimento A**: Los resultados de C-Correcto deberían ser ligeramente superiores a A (gracias a SMOTE en train), pero del mismo orden de magnitud.
- **vs C-Incorrecto**: La diferencia cuantifica el "coste" del leakage en la evaluación del modelo.